# Mundial 2026 – Modelo Predictivo Final v2
> **Cambios en esta versión:**
> - ✅ Fix data leakage: forma reciente calculada con partidos anteriores a cada fecha
> - ✅ Validación ampliada: WC2018, WC2022, Euro2024, Copa América 2024
> - ✅ Comparación de modelos: Elo solo → Elo+ODM → Elo+ODM+Forma → completo
> - ✅ Valor de mercado Transfermarkt integrado como feature
> - ✅ xG como feature adicional del logit (donde disponible)


In [40]:
!py -m pip install kagglehub


   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -------------------------- 1/3 [kagglesdk]
   -------------------------- ------------- 2/3 [kagglehub]
   -------------------------- ------------- 2/3 [kaggl

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [41]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("martj42/international-football-results-from-1872-to-2017")

print("Path to dataset files:", path)

c:\Users\daniel.cejas\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 1.21M/1.21M [00:01<00:00, 663kB/s]

Extracting files...
Path to dataset files: C:\Users\daniel.cejas\.cache\kagglehub\datasets\martj42\international-football-results-from-1872-to-2017\versions\131


## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import os
import math
from scipy.optimize import minimize
from collections import Counter, defaultdict


## 2. Carga de datos

**Rango temporal:**
Cargamos desde 1990 pero el modelo funciona igual desde 2006 o 2010 gracias
al `REVERT_FACTOR`. Puedes cambiar el filtro y comparar RPS de validación
para decidir empíricamente cuál es mejor para tu caso.


In [2]:
# ── Colab: descomenta ────────────────────────────────────────────────────────
# from google.colab import drive; drive.mount('/content/drive')
# DATA_PATH  = "/content/drive/MyDrive/data/results.csv"
# THIRD_PATH = "/content/drive/MyDrive/data/combinaciones_terceros_mundial_2026.csv"

DATA_PATH  = "../data/results.csv"
THIRD_PATH = "../data/combinaciones_terceros_mundial_2026.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["date"])
df = df[df["home_team"] != df["away_team"]].sort_values("date").reset_index(drop=True)

historical_df  = df.dropna(subset=["home_score","away_score"]).copy()
historical_df["home_score"] = historical_df["home_score"].astype(int)
historical_df["away_score"] = historical_df["away_score"].astype(int)
historical_df = historical_df[historical_df["date"] >= "2006-01-01"]

print(f"Partidos históricos: {len(historical_df):,}")
print(f"Rango: {historical_df['date'].min().date()} → {historical_df['date'].max().date()}")


Partidos históricos: 19,550
Rango: 2006-01-02 → 2026-05-31


In [3]:
historical_df

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
29731,2006-01-02,Qatar,Libya,2,0,Friendly,Doha,Qatar,False
29732,2006-01-05,Egypt,Zimbabwe,2,0,Friendly,Alexandria,Egypt,False
29733,2006-01-07,Guinea,Togo,1,0,Friendly,Viry-Châtillon,France,True
29734,2006-01-09,Morocco,DR Congo,3,0,Friendly,Rabat,Morocco,False
29735,2006-01-11,Ghana,Togo,0,1,Friendly,Monastir,Tunisia,True
...,...,...,...,...,...,...,...,...,...
49276,2026-05-31,Germany,Finland,4,0,Friendly,Mainz,Germany,False
49277,2026-05-31,Poland,Ukraine,0,2,Friendly,Wrocław,Poland,False
49278,2026-05-31,Singapore,Mongolia,4,0,Friendly,Singapore,Singapore,False
49279,2026-05-31,Switzerland,Jordan,4,1,Friendly,St. Gallen,Switzerland,False


In [4]:
print(len(historical_df['tournament'].unique()))


113


In [5]:
for tournament, count in historical_df["tournament"].value_counts().items():
    print(tournament, count)

Friendly 6376
FIFA World Cup qualification 4234
UEFA Euro qualification 1322
African Cup of Nations qualification 1182
UEFA Nations League 658
African Cup of Nations 429
CONCACAF Nations League 422
AFC Asian Cup qualification 368
FIFA World Cup 320
Gold Cup 275
COSAFA Cup 255
CECAFA Cup 234
CFU Caribbean Cup qualification 226
UEFA Euro 215
Island Games 204
AFC Asian Cup 198
Copa América 196
AFF Championship 184
Gulf Cup 137
EAFF Championship 104
SAFF Cup 104
CONIFA World Football Cup 101
AFC Challenge Cup qualification 92
AFC Challenge Cup 87
WAFF Championship 78
CFU Caribbean Cup 76
Pacific Games 75
Arab Cup 74
CONCACAF Nations League qualification 68
UNCAF Cup 66
Viva World Cup 60
Indian Ocean Island Games 59
FIFA Series 57
AFF Championship qualification 56
Oceania Nations Cup 56
CONIFA European Football Cup 49
Confederations Cup 48
King's Cup 43
Coupe de l'Outre-Mer 42
CONCACAF Series 35
Gold Cup qualification 34
Nehru Cup 33
Muratti Vase 32
Baltic Cup 31
Kirin Cup 27
ASEAN Champion

In [6]:
a = historical_df[(historical_df['tournament'] == 'FIFA World Cup qualification')]
a.tail(30)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
48965,2025-11-18,Scotland,Denmark,4,2,FIFA World Cup qualification,Glasgow,Scotland,False
48966,2025-11-18,Sweden,Slovenia,1,1,FIFA World Cup qualification,Solna,Sweden,False
48967,2025-11-18,Iraq,United Arab Emirates,2,1,FIFA World Cup qualification,Basra,Iraq,False
48971,2025-11-18,Austria,Bosnia and Herzegovina,1,1,FIFA World Cup qualification,Vienna,Austria,False
48972,2025-11-18,Kosovo,Switzerland,1,1,FIFA World Cup qualification,Pristina,Kosovo,False
48973,2025-11-18,Romania,San Marino,7,1,FIFA World Cup qualification,Ploiești,Romania,False
48974,2025-11-18,Panama,El Salvador,3,0,FIFA World Cup qualification,Panama City,Panama,False
48975,2025-11-18,Wales,North Macedonia,7,1,FIFA World Cup qualification,Cardiff,Wales,False
48976,2025-11-18,Belgium,Liechtenstein,7,0,FIFA World Cup qualification,Liège,Belgium,False
48981,2025-11-18,Guatemala,Suriname,3,1,FIFA World Cup qualification,Guatemala City,Guatemala,False


## 3. Constantes y pesos de competición

In [7]:
INITIAL_ELO   = 1500
INITIAL_ODM   = 1.0
REVERT_FACTOR = 0.99   # regresión a la media por partido

TOURNAMENT_W = {
    "Friendly":                          0.30,
    "FIFA World Cup qualification":      0.55, 
    "UEFA Euro qualification":           0.55, 
    "African Cup of Nations qualification": 0.50,
    "AFC Asian Cup qualification":       0.45,
    "CONCACAF Nations League":           0.45,
    "Gold Cup":                          0.65,
    "AFC Asian Cup":                     0.70,
    "African Cup of Nations":            0.75,
    "UEFA Nations League":               0.80,
    "Copa América":                      0.85,
    "Confederations Cup":                0.80,
    "UEFA Euro":                         0.90,
    "FIFA World Cup":                    1.00,
    "CONMEBOL–UEFA Cup of Champions":    0.90,
    "FIFA Series":                       0.40,
}

def get_w_factor(t):
    return TOURNAMENT_W.get(t, 0.40)


## 4. Sistema Elo con regresión a la media

In [8]:
def expected_score(ra, rb):
    """
    Calcula la probabilidad matemática de victoria del equipo A sobre el equipo B.
    
    Args:
        ra (float): Puntuación Elo actual del equipo A.
        rb (float): Puntuación Elo actual del equipo B.
        
    Returns:
        float: Probabilidad de victoria de A (valor entre 0.0 y 1.0).
    """
    return 1.0 / (1.0 + 10.0 ** ((rb - ra) / 400.0))

def actual_score(hg, ag):
    """
    Convierte el resultado en goles a puntos Elo estándar.
    
    Args:
        hg (int): Goles del equipo local (Home Goals).
        ag (int): Goles del equipo visitante (Away Goals).
        
    Returns:
        tuple: Puntos para el local y visitante (1.0=Victoria, 0.5=Empate, 0.0=Derrota).
    """
    if hg > ag: return 1.0, 0.0
    if hg < ag: return 0.0, 1.0
    return 0.5, 0.5

def update_elo(hr, ar, hg, ag, tournament, k0=40.0, gamma=0.5):
    """
    Actualiza el ranking Elo de ambos equipos tras jugar un partido.
    Aplica regresión a la media, calcula la diferencia entre expectativa y realidad,
    y ajusta los puntos basándose en la importancia del torneo y la goleada.
    
    Args:
        hr (float): Elo actual del local.
        ar (float): Elo actual del visitante.
        hg (int): Goles del local.
        ag (int): Goles del visitante.
        tournament (str): Nombre de la competición.
        k0 (float): Factor base de volatilidad de los puntos.
        gamma (float): Multiplicador que premia ganar por muchos goles.
        
    Returns:
        tuple: Nuevo Elo del equipo local y nuevo Elo del visitante.
    """
    # 1. Regresión a la media (el "efecto goma elástica" del que hablamos antes)
    hr = hr * REVERT_FACTOR + INITIAL_ELO * (1 - REVERT_FACTOR)
    ar = ar * REVERT_FACTOR + INITIAL_ELO * (1 - REVERT_FACTOR)
    
    # 2. Calcular Expectativa vs Realidad
    exp_h = expected_score(hr, ar)
    act_h, act_a = actual_score(hg, ag)
    
    # 3. Calcular la cantidad de puntos en juego (K-factor)
    k = k0 * get_w_factor(tournament) * (1 + abs(hg - ag)) ** gamma
    
    # 4. Asignar los nuevos puntos
    return hr + k*(act_h - exp_h), ar + k*(act_a - (1-exp_h))


## 5. Modelo ODM

In [9]:
def scale(A, x):
    """
    Realiza un paso de ajuste iterativo (tipo PageRank) para balancear los ratings.
    Usa np.maximum con 1e-6 para evitar errores de división por cero si alguien tiene 0 puntos.
    
    Args:
        A (numpy.ndarray): Matriz con los goles del partido.
        x (numpy.ndarray): Vector con los ratings actuales (de ataque o defensa).
        
    Returns:
        tuple: Vectores recalculados en un paso iterativo.
    """
    x_inv  = 1.0 / np.maximum(x, 1e-6)
    y      = A @ x_inv
    y_inv  = 1.0 / np.maximum(y, 1e-6)
    x_post = A.T @ y_inv
    return x_post, A @ (1.0 / np.maximum(x_post, 1e-6))

def compute_post_game_odm(o_pre, d_pre, hg, ag, nb_iter=10, eps=0.01):
    """
    Calcula los nuevos ratings de Ataque (o) y Defensa (d) tras un partido.
    Funciona resolviendo un sistema de ecuaciones de forma iterativa.
    
    Args:
        o_pre (list): Ratings de ataque previos [local, visitante].
        d_pre (list): Ratings de defensa previos [local, visitante].
        hg (int): Goles del local.
        ag (int): Goles del visitante.
        nb_iter (int): Número de iteraciones para que la matemática converja.
        eps (float): Un número minúsculo (0.01) para que el modelo no se rompa en los empates 0-0.
        
    Returns:
        tuple: (nuevo_ataque_local, nuevo_ataque_visitante, nueva_defensa_local, nueva_defensa_visitante)
    """
    # 1. Se crea la matriz del partido con los goles (se suma eps para evitar ceros absolutos)
    A = np.array([[eps, ag+eps], [hg+eps, eps]])
    o_post, d_post = np.array(o_pre, float), np.array(d_pre, float)
    
    # 2. Bucle iterativo donde la matemática "da vueltas" hasta encontrar el equilibrio
    for _ in range(nb_iter):
        o_r_o, d_r_o = scale(A,   o_post)
        d_r_d, o_r_d = scale(A.T, d_post)
        o_post = o_r_o + o_r_d
        d_post = d_r_o + d_r_d
        
    return o_post[0], o_post[1], d_post[0], d_post[1]

## 6. Forma reciente

In [10]:
FORM_MIN_WEIGHT = 0.5

def calculate_recent_form(matches_before_date, team, n=10):
    """
    Calcula la forma reciente usando SOLO partidos ya procesados
    (anteriores a la fecha del partido actual).
    """
    tm = matches_before_date[
        ((matches_before_date["home_team"] == team) |
         (matches_before_date["away_team"] == team)) &
        (matches_before_date["tournament"].apply(get_w_factor) >= FORM_MIN_WEIGHT)
    ].sort_values("date", ascending=False).head(n)

    if len(tm) == 0:
        return 0.0

    scores, weights = [], []
    for i, row in enumerate(tm.itertuples()):
        if row.home_team == team:
            res = 1.0 if row.home_score > row.away_score else (
                  -1.0 if row.home_score < row.away_score else 0.0)
        else:
            res = 1.0 if row.away_score > row.home_score else (
                  -1.0 if row.away_score < row.home_score else 0.0)
        w = math.exp(-0.15 * i) * get_w_factor(row.tournament)
        scores.append(res * w)
        weights.append(w)

    return sum(scores) / sum(weights) if weights else 0.0


## 7. Valor de mercado (Transfermarkt)

**Por qué es una feature valiosa:**
El valor de mercado captura talento individual, edad media de la plantilla
y el momento de forma estructural del equipo — información que el Elo y ODM
no capturan directamente. Groll et al. (2018) lo usan con buenos resultados.

**Limitación importante:**
Solo tenemos valores actuales (2025/2026). Para la validación histórica
(WC2018, WC2022) no podemos usarlo sin datos históricos.
Por tanto, el valor de mercado SOLO entra en los ratings finales para
predecir el 2026, no en el proceso de optimización de parámetros.

**Cómo obtener los datos desde tu máquina local:**
El sandbox donde corre este notebook no puede acceder a Transfermarkt
(bloqueo de IP). Ejecuta el scraper de abajo desde tu entorno local
y guarda el resultado en `../data/market_values.csv`.


In [11]:

MARKET_VALUES_M = {
    "England": 1360,
    "France": 1520,
    "Spain": 1220,
    "Germany": 947,
    "Portugal": 1010,
    "Brazil": 928,
    "Netherlands": 804,
    "Argentina": 782,
    "Belgium": 548,
    "Colombia": 302,
    "United States": 386,
    "Mexico": 192,
    "Uruguay": 359,
    "Japan": 271,
    "Morocco": 498,
    "Croatia": 387,
    "Switzerland": 333,
    "Senegal": 478,
    "Norway": 590,
    "Ecuador": 369,
    "Canada": 197,
    "Turkey": 474,
    "Austria": 242,
    "South Korea": 139,
    "Sweden": 406,
    "Australia": 77,
    "Tunisia": 70,
    "Egypt": 116,
    "Algeria": 257,
    "Ghana": 235,
    "Iran": 32,
    "Saudi Arabia": 41,
    "Paraguay": 154,
    "Panama": 35,
    "New Zealand": 34,
    "Scotland": 170,
    "Iraq": 21,
    "Jordan": 20,
    "South Africa": 49,
    "Qatar": 20, 
    "Czech Republic": 188,
    "Bosnia and Herzegovina": 152,
    "Cape Verde": 55,
    "Haiti": 56,
    "Ivory Coast": 522,
    "DR Congo": 144,
    "Uzbekistan": 85,
    "Curaçao": 26,
}

# Normalizar a escala 0-1 para usar como feature
_max_mv = max(MARKET_VALUES_M.values())
MARKET_VALUES_NORM = {t: v / _max_mv for t, v in MARKET_VALUES_M.items()}

def get_market_value(team):
    """Devuelve el valor de mercado normalizado [0,1]. 0.3 si no hay dato."""
    return MARKET_VALUES_NORM.get(team, 0.3)

print(f"Valores de mercado cargados para {len(MARKET_VALUES_M)} selecciones.")
print(f"Top 5: {sorted(MARKET_VALUES_M.items(), key=lambda x: -x[1])[:5]}")


Valores de mercado cargados para 48 selecciones.
Top 5: [('France', 1520), ('England', 1360), ('Spain', 1220), ('Portugal', 1010), ('Germany', 947)]


## 8. Historial combinado Elo + ODM 

**El cambio clave:**
La forma reciente se calcula DENTRO del bucle cronológico, usando el acumulado
de partidos ya procesados hasta ese momento. Así para un partido de junio 2018,
la forma solo incluye resultados anteriores a junio 2018.

Esto añade coste computacional (hay que filtrar en cada iteración), pero
es el único enfoque metodológicamente correcto.


In [12]:
def calculate_combined_history(matches, k0=40.0, gamma=0.5, decay_days=730,
                                include_form=True):
    """
    Recorre partidos cronológicamente.
    include_form=True: calcula forma sin data leakage (más lento pero correcto).
    include_form=False: omite forma (más rápido, para comparación de modelos).
    """
    ratings_elo = {}
    odm_o_wsum  = defaultdict(float)
    odm_d_wsum  = defaultdict(float)
    odm_w_sum   = defaultdict(float)
    records     = []

    # Acumulado de partidos procesados (para forma sin leakage)
    processed_records = []

    ref_date = matches["date"].max()

    for row in matches.sort_values("date").itertuples():
        home, away = row.home_team, row.away_team
        h_elo = ratings_elo.get(home, INITIAL_ELO)
        a_elo = ratings_elo.get(away, INITIAL_ELO)

        def _odm(t):
            if odm_w_sum[t] > 0:
                return odm_o_wsum[t]/odm_w_sum[t], odm_d_wsum[t]/odm_w_sum[t]
            return INITIAL_ODM, INITIAL_ODM

        h_o, h_d = _odm(home)
        a_o, a_d = _odm(away)

        # ── Forma reciente SIN data leakage ──────────────────────────────────
        if include_form and len(processed_records) > 0:
            proc_df = pd.DataFrame(processed_records)
            h_form = calculate_recent_form(proc_df, home)
            a_form = calculate_recent_form(proc_df, away)
        else:
            h_form, a_form = 0.0, 0.0

        records.append({
            "date": row.date, "home_team": home, "away_team": away,
            "home_score": row.home_score, "away_score": row.away_score,
            "home_elo": h_elo, "away_elo": a_elo,
            "home_o": h_o, "home_d": h_d,
            "away_o": a_o, "away_d": a_d,
            "home_form": h_form, "away_form": a_form,
            "tournament": row.tournament, "neutral": row.neutral,
        })

        # Añadir al acumulado ANTES de procesar el siguiente
        processed_records.append({
            "date": row.date, "home_team": home, "away_team": away,
            "home_score": row.home_score, "away_score": row.away_score,
            "tournament": row.tournament,
        })

        # Actualizar Elo
        new_h, new_a = update_elo(h_elo, a_elo, row.home_score, row.away_score,
                                   row.tournament, k0, gamma)
        ratings_elo[home] = new_h
        ratings_elo[away] = new_a

        # Actualizar ODM con decay temporal
        days_ago   = (ref_date - row.date).days
        time_decay = math.exp(-math.log(2) * days_ago / decay_days)
        wi = get_w_factor(row.tournament) * time_decay
        h_op, a_op, h_dp, a_dp = compute_post_game_odm(
            [h_o, a_o], [h_d, a_d], row.home_score, row.away_score)
        for team, op, dp in [(home, h_op, h_dp), (away, a_op, a_dp)]:
            odm_o_wsum[team] += op * wi
            odm_d_wsum[team] += dp * wi
            odm_w_sum[team]  += wi

    return pd.DataFrame(records, index=matches.sort_values("date").index)


## 9. Métricas de evaluación

In [13]:
def calculate_rps(probs, actual_results):
    """
    Calcula el Ranked Probability Score (RPS). Mide el error de la predicción
    teniendo en cuenta el orden lógico de los resultados (Derrota < Empate < Victoria).
    Cuanto más bajo sea el valor, mejor es el modelo.
    
    Args:
        probs (numpy.ndarray): Matriz de (N, 3) con las probabilidades [Derrota, Empate, Victoria].
        actual_results (numpy.ndarray): Array de longitud N con el resultado real (0, 1 o 2).
        
    Returns:
        float: El valor promedio del RPS para todas las predicciones.
    """
    N = len(actual_results)
    y = np.zeros((N, 3))
    y[np.arange(N), actual_results] = 1.0 # Crea matriz One-Hot para el resultado real
    cp = np.cumsum(probs, axis=1)         # Probabilidad acumulada
    cy = np.cumsum(y,     axis=1)         # Realidad acumulada
    return np.mean(np.sum((cp[:,:2] - cy[:,:2])**2, axis=1) / 2.0)

def calculate_accuracy(probs, actual_results):
    """
    Calcula la precisión bruta (Accuracy). 
    Si la probabilidad más alta coincide con el resultado real, cuenta como acierto.
    
    Args:
        probs (numpy.ndarray): Matriz de probabilidades predichas.
        actual_results (numpy.ndarray): Array con los resultados reales.
        
    Returns:
        float: Porcentaje de aciertos (ej. 0.58 = 58% de acierto).
    """
    return np.mean(np.argmax(probs, axis=1) == actual_results)

def calculate_logloss(probs, actual_results):
    """
    Calcula la pérdida logarítmica (Log-Loss / Cross-Entropy).
    Penaliza fuertemente al modelo si está muy seguro de un resultado falso.
    
    Args:
        probs (numpy.ndarray): Matriz de probabilidades predichas.
        actual_results (numpy.ndarray): Array con los resultados reales.
        
    Returns:
        float: Valor de la penalización (cuanto más cercano a 0, mejor).
    """
    N = len(actual_results)
    p_true = probs[np.arange(N), actual_results] # Coge solo la prob del evento que pasó
    return -np.mean(np.log(np.maximum(p_true, 1e-15))) # 1e-15 evita el colapso matemático log(0)

def encode_result(hg, ag):
    """
    Traduce el marcador de goles a una clase categórica (0, 1, 2) 
    para poder evaluarlo matemáticamente.
    
    Args:
        hg (int): Goles del equipo local (Home Goals).
        ag (int): Goles del equipo visitante (Away Goals).
        
    Returns:
        int: 0 (Derrota local), 1 (Empate), 2 (Victoria local).
    """
    if hg < ag: return 0
    if hg == ag: return 1
    return 2

## 10. Regresión Logit Ordenada

In [14]:
import numpy as np

def ordered_logit_probs(X_beta, theta_0, d_theta):
    """
    Convierte el "Poder Neto" (X_beta) de un partido en probabilidades exactas de 1X2.
    Utiliza Regresión Logística Ordenada para respetar que Derrota < Empate < Victoria.
    
    Args:
        X_beta (numpy.ndarray): Valor numérico que representa la ventaja total del equipo local.
        theta_0 (float): Primer umbral (frontera entre perder y empatar).
        d_theta (float): Distancia logarítmica hasta el segundo umbral.
        
    Returns:
        numpy.ndarray: Matriz con las probabilidades de [Derrota, Empate, Victoria].
    """
    # theta_1 garantiza matemáticamente ser mayor que theta_0 usando np.exp()
    theta_1    = theta_0 + np.exp(d_theta) 
    
    # Probabilidad acumulada (Curva Sigmoide)
    p_loss     = 1.0 / (1.0 + np.exp(-(theta_0 - X_beta)))
    p_loss_tie = 1.0 / (1.0 + np.exp(-(theta_1 - X_beta)))
    
    # Extraemos las probabilidades individuales
    p_tie      = p_loss_tie - p_loss
    p_win      = 1.0 - p_loss_tie
    
    # np.clip evita que la IA dé un 0% absoluto, lo cual rompería las matemáticas
    return np.clip(np.column_stack((p_loss, p_tie, p_win)), 1e-15, 1.0)

def compute_xbeta(df, betas):
    """
    Calcula el "Poder Neto" (X_beta) de cada partido sumando todas las variables 
    (Elo, Ataque, Forma...) multiplicadas por sus pesos (betas).
    
    Args:
        df (pandas.DataFrame): DataFrame con los partidos y sus estadísticas.
        betas (dict): Diccionario con el peso o importancia de cada variable.
        
    Returns:
        numpy.ndarray: Array con la puntuación total de ventaja para el equipo local (X_beta).
    """
    X = np.zeros(len(df))
    
    # Se va sumando la ventaja del equipo local en cada métrica, multiplicada por su peso
    if "beta_elo" in betas:
        X += betas["beta_elo"] * (df["home_elo"] - df["away_elo"]).values / 100.0
    if "beta_odm1" in betas:
        ho = df["home_o_norm"] if "home_o_norm" in df.columns else df["home_o"]
        ad = df["away_d_norm"] if "away_d_norm" in df.columns else df["away_d"]
        X += betas["beta_odm1"] * (ho - ad).values
    if "beta_odm2" in betas:
        hd = df["home_d_norm"] if "home_d_norm" in df.columns else df["home_d"]
        ao = df["away_o_norm"] if "away_o_norm" in df.columns else df["away_o"]
        X += betas["beta_odm2"] * (hd - ao).values
    if "beta_form" in betas:
        X += betas["beta_form"] * (df["home_form"] - df["away_form"]).values
    if "beta_home" in betas:
        X += betas["beta_home"] * (~df["neutral"]).astype(float).values
        
    return X

## 11. Optimización y comparación de modelos

**Qué comparamos:**
Siguiendo la metodología del paper (Tabla 1), evaluamos 4 configuraciones
de features para ver el valor marginal de cada una.

**Conjuntos de validación ampliados:**
- WC2018 (64 partidos) 
- WC2022 (64 partidos)
- UEFA Euro 2024 (51 partidos) ← nuevo
- Copa América 2024 (32 partidos) ← nuevo

Total: ~211 partidos de alta competición reciente.

**Por qué incluir Euro 2024 y Copa América 2024:**
Son los torneos más recientes y del mayor nivel fuera del Mundial.
Validar ahí nos dice si el modelo funciona en competiciones similares
con selecciones en su mejor forma, y con datos muy cercanos al 2026.


In [15]:
## Ejecutar si ya tenemos los datasets:
print("Cargando el historial desde el Checkpoint...")

# 1. Cargamos el historial (IMPORTANTE: le decimos que la columna 'date' es una fecha)
df_hist = pd.read_csv("../data/processed/df_hist.csv", parse_dates=["date"])

# 2. Cargamos el dataset combinado
val_combined = pd.read_csv("../data/processed/val_combined.csv", parse_dates=["date"])

# 3. Cargamos la hoja de respuestas
y_combined = np.load("../data/processed/y_combined.npy")

# 4. Recreamos los diccionarios de torneos (esto es instantáneo porque ya tenemos df_hist)
def val_mask(df, start, end, tournament=None):
    m = (df["date"] >= start) & (df["date"] <= end)
    if tournament:
        m = m & (df["tournament"] == tournament)
    return m

val_sets = {
    "WC2018":    df_hist[val_mask(df_hist,"2018-06-14","2018-07-15","FIFA World Cup")],
    "WC2022":    df_hist[val_mask(df_hist,"2022-11-20","2022-12-18","FIFA World Cup")],
    "Euro2024":  df_hist[val_mask(df_hist,"2024-06-14","2024-07-14","UEFA Euro")],
    "CopaA2024": df_hist[val_mask(df_hist,"2024-06-20","2024-07-14","Copa América")],
}

print(f"¡Listo! Historial cargado con {len(df_hist):,} partidos.")

Cargando el historial desde el Checkpoint...
¡Listo! Historial cargado con 19,550 partidos.


In [22]:
# ejecutar si queremos recalcular los datasets
print("Calculando historial completo (con forma sin data leakage)...")
print("Esto puede tardar varios minutos por el cálculo de forma en cada paso...")
df_hist = calculate_combined_history(historical_df, k0=40.0, gamma=0.5,
                                      include_form=True)
print(f"Historial listo: {len(df_hist):,} partidos")

# ── Índices de validación ─────────────────────────────────────────────────────
def val_mask(df, start, end, tournament=None):
    m = (df["date"] >= start) & (df["date"] <= end)
    if tournament:
        m = m & (df["tournament"] == tournament)
    return m

val_sets = {
    "WC2018":    df_hist[val_mask(df_hist,"2018-06-14","2018-07-15","FIFA World Cup")],
    "WC2022":    df_hist[val_mask(df_hist,"2022-11-20","2022-12-18","FIFA World Cup")],
    "Euro2024":  df_hist[val_mask(df_hist,"2024-06-14","2024-07-14","UEFA Euro")],
    "CopaA2024": df_hist[val_mask(df_hist,"2024-06-20","2024-07-14","Copa América")],
}

for name, vdf in val_sets.items():
    print(f"{name}: {len(vdf)} partidos")

# Combinar todos para optimizar
val_combined = pd.concat(val_sets.values())
y_combined   = np.array([encode_result(r.home_score, r.away_score)
                          for r in val_combined.itertuples()])
print(f"\nTotal validación: {len(val_combined)} partidos")


Calculando historial completo (con forma sin data leakage)...
Esto puede tardar varios minutos por el cálculo de forma en cada paso...
Historial listo: 19,550 partidos
WC2018: 64 partidos
WC2022: 64 partidos
Euro2024: 51 partidos
CopaA2024: 32 partidos

Total validación: 211 partidos


In [16]:
# Recorremos las 4 columnas de estadísticas de Ataque (o) y Defensa (d)
for col in ["home_o", "home_d", "away_o", "away_d"]:
    
    # 1. Buscamos cuál es el valor "Medio / Típico" de toda la historia
    median_val = df_hist[col].median()
    
    # 2. Creamos una NUEVA columna (_norm) dividiendo los puntos de cada equipo entre la media
    df_hist[f"{col}_norm"] = df_hist[col] / median_val

In [17]:
# 1. Calculamos quién llega en mejor racha al partido (Forma Local - Forma Visitante)
# Si da positivo, el local llega mejor. Si da negativo, el visitante llega mejor.
form_diff = val_combined["home_form"] - val_combined["away_form"]

# 2. Traducimos el resultado del partido a una dirección matemática (-1, 0, 1)
# Si y_combined es 2 (victoria local) -> pasa a ser 1
# Si y_combined es 0 (victoria visitante) -> pasa a ser -1
# Si es empate -> pasa a ser 0
y_sign    = np.where(y_combined == 2, 1, np.where(y_combined == 0, -1, 0))

# 3. Calculamos la Correlación de Pearson entre ambas listas
correlation = np.corrcoef(form_diff, y_sign)[0, 1]

# 4. Imprimimos el resultado (en tu notebook dio 0.1446)
print(f"Correlación forma vs resultado: {correlation:.4f}")

Correlación forma vs resultado: 0.2145


Un resultado de 0.2164 indica una correlación positiva y estadísticamente significativa.

Significa que existe una relación real y matemática: cuanto mejor es la racha con la que llegas (Forma), más probabilidades tienes de ganar el partido.

¿Por qué no es un 0.80 o un 0.90? Porque el fútbol es fútbol. Si la racha fuera el único factor para ganar, los partidos serían aburridísimos y siempre ganaría el que viene de golear. Un 0.20 en predicción deportiva (un entorno tan caótico) es un número buenísimo para una sola variable aislada.

In [18]:
os.makedirs("../data/processed", exist_ok=True)

print("Guardando el historial procesado...")

# 1. Guardamos el historial completo
df_hist.to_csv("../data/processed/df_hist.csv", index=False)

# 2. Guardamos el dataset combinado de validación
val_combined.to_csv("../data/processed/val_combined.csv", index=False)

# 3. Guardamos la "hoja de respuestas" (el array de numpy)
np.save("../data/processed/y_combined.npy", y_combined)

print("¡Datos guardados con éxito! Ya no tendrás que recalcularlos la próxima vez.")

Guardando el historial procesado...
¡Datos guardados con éxito! Ya no tendrás que recalcularlos la próxima vez.


In [19]:
# ── Función genérica de optimización ─────────────────────────────────────────
def optimize_model(val_df, y_true, feature_keys):

    def objective_full(params):
        betas   = {k: params[i] for i, k in enumerate(feature_keys)}
        betas["beta_home"] = params[len(feature_keys)]
        theta_0 = params[len(feature_keys) + 1]
        d_theta  = params[len(feature_keys) + 2]
        X   = compute_xbeta(val_df, betas)
        rps = calculate_rps(ordered_logit_probs(X, theta_0, d_theta), y_true)
        # L2: evita betas extremas por overfitting
        lambda_reg = 0.01
        l2 = lambda_reg * sum(params[i]**2 for i in range(len(feature_keys)))
        return rps + l2

    x0 = [0.3] * len(feature_keys) + [0.3, -0.5, 0.0]

    bounds = []
    for k in feature_keys:
        if k in ("beta_elo", "beta_form"):
            bounds.append((0.0, None))   # lógicamente positivos
        else:
            bounds.append((None, None))  # beta_odm1/2 pueden ser negativos
    bounds += [(0.0, None), (None, None), (-5.0, 5.0)]  # beta_home, theta_0, d_theta

    res = minimize(objective_full, x0=x0, method="L-BFGS-B", bounds=bounds,
                   options={"maxiter": 300, "disp": False})

    best_betas = {k: res.x[i] for i, k in enumerate(feature_keys)}
    best_betas["beta_home"] = res.x[len(feature_keys)]
    return best_betas, res.x[len(feature_keys)+1], res.x[len(feature_keys)+2], res.fun

# ── Comparación de 4 modelos ──────────────────────────────────────────────────
model_configs = {
    "M1: Solo Elo":           ["beta_elo"],
    "M2: Elo + ODM":          ["beta_elo", "beta_odm1", "beta_odm2"],
    "M3: Elo + ODM + Forma":  ["beta_elo", "beta_odm1", "beta_odm2", "beta_form"],
    #"M4: Completo + MktVal":  ["beta_elo", "beta_odm1", "beta_odm2", "beta_form", "beta_mv"],
}

comparison_results = []
print("Comparando modelos...\n")

for model_name, features in model_configs.items():
    #betas, theta_0, d_theta, rps_val = optimize_model(val_combined, y_combined, features, len(features))
    betas, theta_0, d_theta, rps_val = optimize_model(val_combined, y_combined, features)

    # Evaluar en cada conjunto de validación por separado
    row = {"Modelo": model_name, "RPS_total": rps_val}
    for vname, vdf in val_sets.items():
        if len(vdf) == 0:
            row[f"RPS_{vname}"] = None
            continue
        y_v = np.array([encode_result(r.home_score, r.away_score) for r in vdf.itertuples()])
        X_v = compute_xbeta(vdf, {**betas, "beta_home": betas.get("beta_home",0)})
        probs_v = ordered_logit_probs(X_v, theta_0, d_theta)
        row[f"RPS_{vname}"]  = calculate_rps(probs_v, y_v)
        row[f"Acc_{vname}"]  = calculate_accuracy(probs_v, y_v)

    comparison_results.append(row)
    print(f"{model_name}")
    print(f"  RPS total validación: {rps_val:.4f}")
    print(f"  Betas: { {k:round(v,3) for k,v in betas.items()} }")
    print()

comp_df = pd.DataFrame(comparison_results)
print("\n=== TABLA COMPARATIVA COMPLETA (RPS y Accuracy) ===")

# Seleccionamos todas las columnas en un orden lógico para que sea fácil de leer
columnas_todas = [
    "Modelo", "RPS_total", 
    "RPS_WC2018", "Acc_WC2018", 
    "RPS_WC2022", "Acc_WC2022", 
    "RPS_Euro2024", "Acc_Euro2024", 
    "RPS_CopaA2024", "Acc_CopaA2024"
]

print(comp_df[columnas_todas].to_string(index=False))


Comparando modelos...

M1: Solo Elo
  RPS total validación: 0.2088
  Betas: {'beta_elo': np.float64(0.567), 'beta_home': np.float64(0.0)}



C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_5904\243772410.py:26: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  res = minimize(objective_full, x0=x0, method="L-BFGS-B", bounds=bounds,
C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_5904\243772410.py:26: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  res = minimize(objective_full, x0=x0, method="L-BFGS-B", bounds=bounds,


M2: Elo + ODM
  RPS total validación: 0.2042
  Betas: {'beta_elo': np.float64(0.41), 'beta_odm1': np.float64(0.006), 'beta_odm2': np.float64(0.001), 'beta_home': np.float64(0.0)}



C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_5904\243772410.py:26: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  res = minimize(objective_full, x0=x0, method="L-BFGS-B", bounds=bounds,


M3: Elo + ODM + Forma
  RPS total validación: 0.2042
  Betas: {'beta_elo': np.float64(0.41), 'beta_odm1': np.float64(0.006), 'beta_odm2': np.float64(0.001), 'beta_form': np.float64(0.0), 'beta_home': np.float64(0.0)}


=== TABLA COMPARATIVA COMPLETA (RPS y Accuracy) ===
               Modelo  RPS_total  RPS_WC2018  Acc_WC2018  RPS_WC2022  Acc_WC2022  RPS_Euro2024  Acc_Euro2024  RPS_CopaA2024  Acc_CopaA2024
         M1: Solo Elo   0.208769    0.216162     0.50000    0.224896    0.468750      0.196394      0.509804       0.160231        0.62500
        M2: Elo + ODM   0.204178    0.223764     0.53125    0.240682    0.421875      0.208302      0.411765       0.183407        0.53125
M3: Elo + ODM + Forma   0.204178    0.223769     0.53125    0.240691    0.421875      0.208308      0.411765       0.183413        0.53125


In [20]:
# ── Guardar el mejor modelo para la simulación ────────────────────────────────
# El mejor modelo es el que tiene menor RPS total.


best_row = comp_df.loc[comp_df["RPS_total"].idxmin()]
best_model_name = best_row["Modelo"]
print(f"Mejor modelo: {best_model_name} (RPS={best_row['RPS_total']:.4f})")

# Re-optimizar el mejor modelo para obtener parámetros limpios
best_features = model_configs[best_model_name]
#best_betas, best_theta0, best_dtheta, _ = optimize_model(val_combined, y_combined, best_features, len(best_features))
best_betas, best_theta0, best_dtheta, _ = optimize_model(val_combined, y_combined, best_features)

optimal_params = {
    "k0": 40.0, "gamma": 0.5,
    **best_betas,
    "theta_0": best_theta0,
    "d_theta": best_dtheta,
}

# Asegurar que todos los campos existen (con 0 si no están en el mejor modelo)
for key in ["beta_elo","beta_odm1","beta_odm2","beta_form","beta_mv","beta_home"]:
    optimal_params.setdefault(key, 0.0)

print("\nParámetros óptimos del mejor modelo:")
for k, v in optimal_params.items():
    print(f"  {k:<14} = {v:.6f}")


Mejor modelo: M2: Elo + ODM (RPS=0.2042)


C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_5904\243772410.py:26: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  res = minimize(objective_full, x0=x0, method="L-BFGS-B", bounds=bounds,



Parámetros óptimos del mejor modelo:
  k0             = 40.000000
  gamma          = 0.500000
  beta_elo       = 0.409972
  beta_odm1      = 0.005629
  beta_odm2      = 0.001304
  beta_home      = 0.000000
  theta_0        = -0.143438
  d_theta        = 0.152024
  beta_form      = 0.000000
  beta_mv        = 0.000000


Parámetros:
Parámetros óptimos del mejor modelo:
  k0             = 40.000000
  gamma          = 0.500000
  beta_elo       = 0.384682
  beta_odm1      = 0.005506
  beta_odm2      = 0.001400
  beta_home      = 0.000000
  theta_0        = -0.157992
  d_theta        = 0.144231
  beta_form      = 0.000000
  beta_mv        = 0.000000

  Tras simular matemáticamente torneos pasados, nuestra Inteligencia Artificial ha destilado la fórmula del Mundial: el 95% del peso predictivo recae en el rendimiento histórico sostenido (Sistema Elo), pero en los cruces igualados, el modelo detecta que la eficacia ofensiva tiene un impacto 4 veces mayor que la solidez defensiva. Además, la IA ha depurado automáticamente factores como la localía, ya que en un Mundial el césped es neutral para casi todos. Tenemos un modelo puramente táctico y de rendimiento, sin ruido emocional.

## 12. Grupos del Mundial 2026

In [21]:
groups = {
    "A": ["Mexico","South Africa","South Korea","Czech Republic"],
    "B": ["Canada","Bosnia and Herzegovina","Qatar","Switzerland"],
    "C": ["Brazil","Morocco","Haiti","Scotland"],
    "D": ["United States","Paraguay","Australia","Turkey"],
    "E": ["Germany","Curaçao","Ivory Coast","Ecuador"],
    "F": ["Netherlands","Japan","Sweden","Tunisia"],
    "G": ["Belgium","Egypt","Iran","New Zealand"],
    "H": ["Spain","Cape Verde","Saudi Arabia","Uruguay"],
    "I": ["France","Senegal","Iraq","Norway"],
    "J": ["Argentina","Algeria","Austria","Jordan"],
    "K": ["Portugal","DR Congo","Uzbekistan","Colombia"],
    "L": ["England","Croatia","Ghana","Panama"],
}
third_mapping_df = pd.read_csv(THIRD_PATH)


## 13. Ratings finales

In [22]:
# 1. Hacemos una lista plana con los equipos que SÍ van al Mundial (sacados de tus grupos)
equipos_mundial = [equipo for sublista in groups.values() for equipo in sublista]

latest_ratings = {}

# 2. Solo calculamos el rating para los equipos que van al Mundial
for team in equipos_mundial:
    mask = (df_hist["home_team"] == team) | (df_hist["away_team"] == team)
    
    historial_equipo = df_hist[mask]
    
    if len(historial_equipo) == 0:
        continue # Si el equipo no tiene datos, lo saltamos
        
    last = historial_equipo.iloc[-1]
    
    # Recalculamos para obtener el Elo DESPUÉS del último partido
    h_elo_post, a_elo_post = update_elo(
        last["home_elo"], last["away_elo"], 
        last["home_score"], last["away_score"], 
        last["tournament"], optimal_params["k0"], optimal_params["gamma"]
    )
    
    if last["home_team"] == team:
        r = {"elo": h_elo_post, "o": last["home_o"], "d": last["home_d"], "form": last["home_form"]}
    else:
        r = {"elo": a_elo_post, "o": last["away_o"], "d": last["away_d"], "form": last["away_form"]}
        
    r["mv"] = get_market_value(team)
    latest_ratings[team] = r

# 3. Normalizamos Ataque y Defensa (solo comparando a los equipos del Mundial)
odm_o_median = np.median([v["o"] for v in latest_ratings.values()])
odm_d_median = np.median([v["d"] for v in latest_ratings.values()])

for team in latest_ratings:
    latest_ratings[team]["o_norm"] = latest_ratings[team]["o"] / odm_o_median
    latest_ratings[team]["d_norm"] = latest_ratings[team]["d"] / odm_d_median

# 4. Creamos el DataFrame final, ahora SÍ usando los valores normalizados
ranking_df = (
    pd.DataFrame([{"team":t,"elo":v["elo"],"attack":v["o_norm"],"defense":v["d_norm"],
                   "form":v["form"],"market_value_M":MARKET_VALUES_M.get(t,0)}
                  for t,v in latest_ratings.items()])
    .sort_values("elo", ascending=False).reset_index(drop=True)
)

# Guardamos el resultado limpio
ranking_df.to_csv("../data/processed/ranking_final_2026.csv", index=False)

print("TOP 15 MUNDIALISTAS:")
print(ranking_df.head(15).to_string(index=False))

TOP 15 MUNDIALISTAS:
       team         elo   attack  defense     form  market_value_M
      Spain 1879.392948 2.485377 0.313362 0.610331            1220
    Morocco 1807.898122 1.657232 0.619785 0.750104             498
  Argentina 1807.234855 3.647580 0.337163 0.336605             782
     France 1794.437103 1.861400 0.557502 0.593099            1520
    England 1776.709633 1.727491 0.498023 1.000000            1360
      Japan 1766.980427 1.682633 0.676250 0.485626             271
   Portugal 1746.385441 1.410739 0.669131 0.383447            1010
    Germany 1736.505169 1.438746 0.660256 0.419746             947
    Senegal 1734.231425 1.143253 0.782272 0.536533             478
Netherlands 1731.682386 1.689067 0.512533 0.631645             804
     Brazil 1719.477716 2.948007 0.363476 0.194632             928
    Algeria 1716.902837 1.063142 0.934541 0.557536             257
     Norway 1715.647080 0.706376 1.478422 1.000000             590
       Iran 1714.227953 1.516550 0.619485

## 14. Predicción y simulación

In [23]:
MV_BETA_FIXED = 0.3  # beta fijo, no optimizado — evita data leakage temporal

def predict_match(home, away, ratings, params, neutral=True):
    h, a = ratings[home], ratings[away]

    elo_diff  = (h["elo"] - a["elo"]) / 100.0
    odm_o_d   =  h.get("o_norm", h["o"]) - a.get("d_norm", a["d"])
    odm_d_o   =  h.get("d_norm", h["d"]) - a.get("o_norm", a["o"])
    form_diff =  h.get("form", 0.0) - a.get("form", 0.0)
    mv_diff   =  h.get("mv", 0.3)   - a.get("mv", 0.3)
    home_adv  = 0.0 if neutral else 1.0

    X = (params.get("beta_elo",  0) * elo_diff  +
         params.get("beta_odm1", 0) * odm_o_d   +
         params.get("beta_odm2", 0) * odm_d_o   +
         params.get("beta_form", 0) * form_diff +
         MV_BETA_FIXED               * mv_diff   +
         params.get("beta_home", 0) * home_adv)
    

    probs = ordered_logit_probs(np.array([X]), params["theta_0"], params["d_theta"])[0]
    return {"away_win": probs[0], "draw": probs[1], "home_win": probs[2]}


In [24]:
import numpy as np

def dixon_coles_tau(gh, ga, lh, la, rho=-0.13):
    """
    Aplica el factor de corrección de Dixon-Coles para distribuciones Poisson.
    Corrige la tendencia matemática a subestimar empates y resultados bajos (0-0, 1-0, 0-1, 1-1).
    
    Args:
        gh (int): Goles del equipo local (home).
        ga (int): Goles del equipo visitante (away).
        lh (float): Goles esperados del local (lambda).
        la (float): Goles esperados del visitante (lambda).
        rho (float): Parámetro de dependencia (fijo en -0.13 estándar en fútbol).
        
    Returns:
        float: Multiplicador de probabilidad (factor de ajuste).
    """
    if gh==0 and ga==0: return 1-lh*la*rho
    elif gh==0 and ga==1: return 1+lh*rho
    elif gh==1 and ga==0: return 1+la*rho
    elif gh==1 and ga==1: return 1-rho
    return 1.0

def sample_score(home, away, ratings, result, params=None, rho=-0.13):
    h, a = ratings[home], ratings[away]
    elo_diff = (h["elo"] - a["elo"]) / 100.0
    # Medias ancladas a la realidad: ~1.35 goles base, moduladas por Elo
    lh = min(max(math.exp(0.30 + 0.18 * elo_diff), 0.3), 3.5)
    la = min(max(math.exp(0.10 - 0.18 * elo_diff), 0.3), 3.5)
    for _ in range(500):
        gh = np.random.poisson(lh)
        ga = np.random.poisson(la)
        if np.random.random() > dixon_coles_tau(gh, ga, lh, la, rho):
            continue
        if result == "home_win" and gh > ga: return gh, ga
        if result == "away_win" and ga > gh: return gh, ga
        if result == "draw"     and gh == ga: return gh, ga
    if result == "home_win": return 1, 0
    if result == "away_win": return 0, 1
    return 0, 0



In [25]:
def simulate_group_match(home, away, ratings, params):
    """
    Simula un partido de fase de grupos donde el empate SÍ es posible.
    Lanza unos dados virtuales trucados con las probabilidades de cada equipo.
    
    Args:
        home (str): Equipo local.
        away (str): Equipo visitante.
        ratings (dict): Poderes actuales.
        params (dict): Betas del modelo.
        
    Returns:
        str: El resultado aleatorio ("home_win", "draw", "away_win").
    """
    p = predict_match(home, away, ratings, params, neutral=True)
    return np.random.choice(
        ["away_win", "draw", "home_win"],
        p=[p["away_win"], p["draw"], p["home_win"]]
    )

def simulate_knockout_match(ta, tb, ratings, params):
    """
    Simula un partido eliminatorio (octavos, cuartos...) donde NO hay empate.
    Si el modelo predice empate en 90 min, reparte la probabilidad al 50/50 
    simulando una prórroga o penaltis.
    
    Args:
        ta (str): Equipo A.
        tb (str): Equipo B.
        ratings (dict): Poderes actuales.
        params (dict): Betas del modelo.
        
    Returns:
        str: El nombre del equipo que avanza a la siguiente ronda.
    """
    p = predict_match(ta, tb, ratings, params, neutral=True)
    
    # Repartimos la probabilidad de empate a partes iguales (Penaltis)
    pa = p["home_win"] + 0.5 * p["draw"]
    pb = p["away_win"] + 0.5 * p["draw"]
    
    total = pa + pb
    return np.random.choice([ta, tb], p=[pa/total, pb/total])

In [38]:
# 1. Simulación completa: España vs Portugal

# 1. Predicción: España vs Cape Verde (Un derbi ibérico en campo neutral)
pred_esp_por = predict_match("Spain", "Cape Verde", latest_ratings, optimal_params, neutral=True)

print("🏆 === ESPAÑA vs Cape Verde ===")
print(f"Victoria España:   {pred_esp_por['home_win'] * 100:.1f}%")
print(f"Empate:            {pred_esp_por['draw'] * 100:.1f}%")
print(f"Victoria Cape Verde: {pred_esp_por['away_win'] * 100:.1f}%\n")

res_esp_por = simulate_group_match("Spain", "Cape Verde", latest_ratings, optimal_params)
goles_esp, goles_por = sample_score("Spain", "Cape Verde", latest_ratings, res_esp_por)

print(f"El dado de Montecarlo ha decidido: '{res_esp_por}'")
print(f"Marcador final: España {goles_esp} - {goles_por} Cape Verde\n")


# 2. Simulación completa: Noruega vs Irán

# 2. Predicción: Noruega vs Irán (El poder de Haaland contra la resistencia iraní)
pred_nor_ira = predict_match("Argentina", "Austria", latest_ratings, optimal_params, neutral=True)

print("🌍 === ARGENTINA vs AUSTRIA ===")
print(f"Victoria Argentina:  {pred_nor_ira['home_win'] * 100:.1f}%")
print(f"Empate:            {pred_nor_ira['draw'] * 100:.1f}%")
print(f"Victoria Austria:     {pred_nor_ira['away_win'] * 100:.1f}%")

res_nor_ira = simulate_group_match("Argentina", "Austria", latest_ratings, optimal_params)
goles_nor, goles_ira = sample_score("Argentina", "Austria", latest_ratings, res_nor_ira)

print(f"El dado de Montecarlo ha decidido: '{res_nor_ira}'")
print(f"Marcador final: Argentina {goles_nor} - {goles_ira} Austria")

🏆 === ESPAÑA vs Cape Verde ===
Victoria España:   60.4%
Empate:            22.6%
Victoria Cape Verde: 17.0%

El dado de Montecarlo ha decidido: 'home_win'
Marcador final: España 3 - 0 Cape Verde

🌍 === ARGENTINA vs AUSTRIA ===
Victoria Argentina:  43.0%
Empate:            27.7%
Victoria Austria:     29.3%
El dado de Montecarlo ha decidido: 'draw'
Marcador final: Argentina 0 - 0 Austria


In [31]:
pred_nor_ira = predict_match("Spain", "Portugal", latest_ratings, optimal_params, neutral=True)

print("🌍 === Spain vs Porugal ===")
print(f"Victoria Spain:  {pred_nor_ira['home_win'] * 100:.1f}%")
print(f"Empate:            {pred_nor_ira['draw'] * 100:.1f}%")
print(f"Victoria Portugal:     {pred_nor_ira['away_win'] * 100:.1f}%")

res_nor_ira = simulate_group_match("Spain", "Portugal", latest_ratings, optimal_params)
goles_nor, goles_ira = sample_score("Spain", "Portugal", latest_ratings, res_nor_ira)

print(f"El dado de Montecarlo ha decidido: '{res_nor_ira}'")
print(f"Marcador final: Spain {goles_nor} - {goles_ira} Portugal")

🌍 === Spain vs Porugal ===
Victoria Spain:  39.5%
Empate:            28.2%
Victoria Portugal:     32.3%
El dado de Montecarlo ha decidido: 'home_win'
Marcador final: Spain 2 - 0 Portugal


In [34]:
def simulate_group(teams, ratings, params):
    table = {t:{"pts":0,"gf":0,"ga":0,"gd":0} for t in teams}
    for i in range(len(teams)):
        for j in range(i+1,len(teams)):
            home,away = teams[i],teams[j]
            res = simulate_group_match(home,away,ratings,params)
            hg,ag = sample_score(home,away,ratings,res)
            t=table
            t[home]["gf"]+=hg; t[home]["ga"]+=ag
            t[away]["gf"]+=ag; t[away]["ga"]+=hg
            t[home]["gd"]=t[home]["gf"]-t[home]["ga"]
            t[away]["gd"]=t[away]["gf"]-t[away]["ga"]
            if hg>ag: t[home]["pts"]+=3
            elif ag>hg: t[away]["pts"]+=3
            else: t[home]["pts"]+=1; t[away]["pts"]+=1
    return sorted(table.items(),
        key=lambda x:(x[1]["pts"],x[1]["gd"],x[1]["gf"],ratings[x[0]]["elo"]),reverse=True)

def simulate_group_stage(groups, ratings, params):
    winners,runners,thirds={},{},[]
    for name,teams in groups.items():
        rank=simulate_group(teams,ratings,params)
        winners[name]=rank[0][0]; runners[name]=rank[1][0]
        thirds.append({"group":name,"team":rank[2][0],"pts":rank[2][1]["pts"],
                        "gd":rank[2][1]["gd"],"gf":rank[2][1]["gf"]})
    return {"winners":winners,"runners_up":runners,"thirds":thirds}

def select_best_thirds(thirds, ratings):
    return sorted(thirds,key=lambda x:(x["pts"],x["gd"],x["gf"],
                  ratings[x["team"]]["elo"]),reverse=True)[:8]

def get_third_place_mapping(best_thirds, mapping_df):
    qs=",".join(sorted(x["group"] for x in best_thirds))
    row=mapping_df[mapping_df["Grupos_Clasificados"]==qs]
    if len(row)==0: raise ValueError(f"Sin mapeo: {qs}")
    return row.iloc[0]

def build_round_of_32(gs,best_thirds,mapping_row):
    W=gs["winners"]; R=gs["runners_up"]
    T={x["group"]:x["team"] for x in best_thirds}
    def t(col): return T[mapping_row[col][1]]
    return {73:(R["A"],R["B"]),74:(W["E"],t("Rival_de_1E")),75:(W["F"],R["C"]),
            76:(W["C"],R["F"]),77:(W["I"],t("Rival_de_1I")),78:(R["E"],R["I"]),
            79:(W["A"],t("Rival_de_1A")),80:(W["L"],t("Rival_de_1L")),
            81:(W["D"],t("Rival_de_1D")),82:(W["G"],t("Rival_de_1G")),
            83:(R["K"],R["L"]),84:(W["H"],R["J"]),85:(W["B"],t("Rival_de_1B")),
            86:(W["J"],R["H"]),87:(W["K"],t("Rival_de_1K")),88:(R["D"],R["G"])}

def build_round_of_16(w32):
    return {89:(w32[74],w32[77]),90:(w32[73],w32[75]),91:(w32[76],w32[78]),92:(w32[79],w32[80]),
            93:(w32[83],w32[84]),94:(w32[81],w32[82]),95:(w32[86],w32[88]),96:(w32[85],w32[87])}

def build_quarters(w16):
    return {97:(w16[89],w16[90]),98:(w16[93],w16[94]),99:(w16[91],w16[92]),100:(w16[95],w16[96])}

def build_semis(w8): return {101:(w8[97],w8[98]),102:(w8[99],w8[100])}
def build_final(w4): return {104:(w4[101],w4[102])}

def play_match_dict(matches,ratings,params):
    return {mid:simulate_knockout_match(ta,tb,ratings,params) for mid,(ta,tb) in matches.items()}

def simulate_world_cup(groups,ratings,params,mapping_df):
    gs=simulate_group_stage(groups,ratings,params)
    thirds=select_best_thirds(gs["thirds"],ratings)
    mapping=get_third_place_mapping(thirds,mapping_df)
    w32=play_match_dict(build_round_of_32(gs,thirds,mapping),ratings,params)
    w16=play_match_dict(build_round_of_16(w32),ratings,params)
    w8=play_match_dict(build_quarters(w16),ratings,params)
    w4=play_match_dict(build_semis(w8),ratings,params)
    return play_match_dict(build_final(w4),ratings,params)[104]


## 15. Simulación Monte Carlo para la probabilidad de ganador

In [35]:
def simulate_world_cup_all_stages(groups, ratings, params, mapping_df):
    """
    Juega el Mundial pero guarda el registro de quién alcanza cada fase.
    """
    gs = simulate_group_stage(groups, ratings, params)
    thirds = select_best_thirds(gs["thirds"], ratings)
    mapping = get_third_place_mapping(thirds, mapping_df)

    # 1. Dieciseisavos (Los 32 clasificados)
    matches_32 = build_round_of_32(gs, thirds, mapping)
    teams_r32 = [t for match in matches_32.values() for t in match]

    # 2. Octavos (Los ganadores de R32)
    w32 = play_match_dict(matches_32, ratings, params)
    teams_r16 = list(w32.values())

    # 3. Cuartos (Los ganadores de R16)
    w16 = play_match_dict(build_round_of_16(w32), ratings, params)
    teams_qf = list(w16.values())

    # 4. Semifinales (Los ganadores de QF)
    w8 = play_match_dict(build_quarters(w16), ratings, params)
    teams_sf = list(w8.values())

    # 5. Final (Los ganadores de SF)
    w4 = play_match_dict(build_semis(w8), ratings, params)
    teams_final = list(w4.values())

    # 6. Campeón (El ganador de la Final)
    champion = play_match_dict(build_final(w4), ratings, params)[104]

    return {
        "Dieciseisavos": teams_r32,
        "Octavos": teams_r16,
        "Cuartos": teams_qf,
        "Semis": teams_sf,
        "Final": teams_final,
        "Campeón": [champion]
    }

from collections import defaultdict, Counter
import pandas as pd
import os

N = 50_000
print(f"Iniciando {N:,} simulaciones para Matriz de Supervivencia...")

# Diccionario mágico que cuenta cuántas veces llega cada equipo a cada fase
stage_counters = defaultdict(Counter)

for i in range(N):
    stages = simulate_world_cup_all_stages(groups, latest_ratings, optimal_params, third_mapping_df)
    
    # Sumar +1 a cada equipo en la fase correspondiente
    for stage_name, teams in stages.items():
        for team in teams:
            stage_counters[stage_name][team] += 1
            
    if (i+1) % 5000 == 0: 
        print(f"  {i+1:,} / {N:,} simulaciones completadas")

# --- Construir el DataFrame Final ---
all_teams = [t for group_teams in groups.values() for t in group_teams]
df_survival = pd.DataFrame(index=all_teams)
fases_orden = ["Dieciseisavos", "Octavos", "Cuartos", "Semis", "Final", "Campeón"]

for fase in fases_orden:
    df_survival[fase] = df_survival.index.map(lambda t: stage_counters[fase].get(t, 0) / N)

df_survival = df_survival.sort_values(by=fases_orden[::-1], ascending=False)

# Exportamos la nueva Matriz Completa
os.makedirs("../data/processed", exist_ok=True)
csv_survival_path = "../data/processed/survival_probabilities.csv"
df_survival.reset_index(names="Team").to_csv(csv_survival_path, index=False)

# EXPORTACIÓN DE COMPATIBILIDAD PARA TU STREAMLIT (mc_results.csv)
mc_results = df_survival.reset_index(names="team")[["team", "Campeón"]].rename(columns={"Campeón": "probability"})
mc_results.to_csv("../data/processed/mc_results.csv", index=False)

# Imprimir en consola
print("\n=== MATRIZ DE SUPERVIVENCIA MUNDIAL 2026 (TOP 15) ===")
df_print = df_survival.head(15).copy()
for col in df_print.columns:
    df_print[col] = df_print[col].map(lambda x: f"{x*100:.1f}%")
print(df_print.to_string())
print(f"\n✅ Archivos guardados con éxito en la carpeta /processed/")

Iniciando 50,000 simulaciones para Matriz de Supervivencia...
  5,000 / 50,000 simulaciones completadas
  10,000 / 50,000 simulaciones completadas
  15,000 / 50,000 simulaciones completadas
  20,000 / 50,000 simulaciones completadas
  25,000 / 50,000 simulaciones completadas
  30,000 / 50,000 simulaciones completadas
  35,000 / 50,000 simulaciones completadas
  40,000 / 50,000 simulaciones completadas
  45,000 / 50,000 simulaciones completadas
  50,000 / 50,000 simulaciones completadas

=== MATRIZ DE SUPERVIVENCIA MUNDIAL 2026 (TOP 15) ===
            Dieciseisavos Octavos Cuartos  Semis  Final Campeón
Spain               91.8%   61.1%   44.6%  28.3%  21.4%   13.4%
England             81.4%   50.1%   30.5%  19.5%  11.5%    7.1%
France              75.6%   50.9%   36.0%  20.6%  11.6%    6.7%
Argentina           77.7%   46.2%   29.5%  16.7%  11.2%    6.4%
Morocco             85.4%   51.6%   29.5%  18.2%   8.8%    4.9%
Portugal            68.2%   36.2%   21.4%  12.9%   7.4%    4.2%
Japan 

In [36]:
import pandas as pd

# 1. Preparamos el ranking de Expectativa (Valor de Mercado)
market_df = pd.DataFrame(list(MARKET_VALUES_M.items()), columns=["team", "market_value"])
market_df["rank_mercado"] = market_df["market_value"].rank(ascending=False, method="min")

# 2. Preparamos el ranking de tu Modelo (Probabilidades Monte Carlo)
# Si un equipo tiene 0%, lo ponemos al final de la tabla
mc_df = mc_results.copy()
mc_df["rank_ia"] = mc_df["probability"].rank(ascending=False, method="min")

# 3. Cruzamos los datos
analisis_df = pd.merge(market_df, mc_df, on="team", how="left")
analisis_df["probability"] = analisis_df["probability"].fillna(0)

# Para los que tienen 0% de ganar, les asignamos el último rank
max_rank = analisis_df["rank_ia"].max() + 1
analisis_df["rank_ia"] = analisis_df["rank_ia"].fillna(max_rank)

# 4. LA FÓRMULA MÁGICA (El Delta)
analisis_df["delta_sorpresa"] = analisis_df["rank_mercado"] - analisis_df["rank_ia"]

# Ordenamos por la mayor revelación (Delta positivo más alto)
analisis_df = analisis_df.sort_values("delta_sorpresa", ascending=False).reset_index(drop=True)

print("🌟 TOP 3 REVELACIONES (Dark Horses) 🌟")
print("Equipos que llegarán mucho más lejos de lo que su valor indica:")
display(analisis_df.head(3)[["team", "market_value", "probability", "delta_sorpresa"]])

print("\n📉 TOP 3 DECEPCIONES (Flops) 📉")
print("Equipos millonarios que la IA predice que se estrellarán:")
display(analisis_df.tail(3)[["team", "market_value", "probability", "delta_sorpresa"]])

🌟 TOP 3 REVELACIONES (Dark Horses) 🌟
Equipos que llegarán mucho más lejos de lo que su valor indica:


,team,market_value,probability,delta_sorpresa
0,Iran,32,0.02734,34.0
1,Panama,35,0.01360,18.0
2,Australia,77,0.01988,16.0



📉 TOP 3 DECEPCIONES (Flops) 📉
Equipos millonarios que la IA predice que se estrellarán:


,team,market_value,probability,delta_sorpresa
45,Bosnia and Herzegovina,152,0.00088,-17.0
46,Belgium,548,0.01232,-18.0
47,Sweden,406,0.00398,-23.0


# 16 Simulación Monte Carlo para toda la fase

In [37]:
import json
from collections import Counter
import numpy as np

def get_match_consensus(home, away, ratings, params, is_knockout=False, num_sims=10000):
    # ── Probabilidades analiticas exactas (sin ruido de muestreo) ──────────────
    p = predict_match(home, away, ratings, params, neutral=True)

    # best_1X2 sale de las probabilidades exactas, NO de contar simulaciones.
    # Asi evitamos que el ruido elija mal cuando home/away estan a una milesima.
    best_1X2 = max(p, key=p.get)

    # Marca de partido parejo: si el 1o y el 2o resultado estan a menos de 5 pts,
    # el partido esta "muy abierto" y la app puede avisarlo en vez de fingir favorito.
    ordered = sorted(p.values(), reverse=True)
    is_tight = bool((ordered[0] - ordered[1]) < 0.05)

    # ── Simular solo para el marcador de consenso ──────────────────────────────
    outcomes, scores = [], []
    for _ in range(num_sims):
        res = np.random.choice(
            ["away_win", "draw", "home_win"],
            p=[p["away_win"], p["draw"], p["home_win"]]
        )
        outcomes.append(res)
        scores.append(sample_score(home, away, ratings, res))

    result = {
        "home": home, "away": away,
        "probs_1X2": {
            "home_win": round(p["home_win"], 3),
            "draw":     round(p["draw"], 3),
            "away_win": round(p["away_win"], 3),
        },
        "best_1X2": best_1X2,
        "is_tight": is_tight,
    }

    if not is_knockout:
        # ── FASE DE GRUPOS ─────────────────────────────────────────────────────
        # Marcador = el mas frecuente DENTRO del resultado mas probable.
        # Vale igual para victoria o empate (si best_1X2 es draw, coge el empate
        # mas frecuente, tipicamente 1-1). Asi marcador y resultado siempre concuerdan.
        valid = [s for s, r in zip(scores, outcomes) if r == best_1X2]
        if valid:
            hg, ag = Counter(valid).most_common(1)[0][0]
        else:
            hg, ag = {"home_win": (1, 0), "away_win": (0, 1), "draw": (1, 1)}[best_1X2]
        result["score"] = f"{hg}-{ag}"

    else:
        # ── ELIMINATORIA ───────────────────────────────────────────────────────
        # Quien avanza: exacto, sin ruido (victoria + mitad de los empates).
        adv_home = p["home_win"] + 0.5 * p["draw"]
        adv_away = p["away_win"] + 0.5 * p["draw"]
        if adv_home >= adv_away:
            advancer, advances_prob = home, adv_home
        else:
            advancer, advances_prob = away, adv_away
        result["advances"] = advancer
        result["advances_prob"] = round(advances_prob, 3)

        # Marcador COHERENTE con quien avanza (Opcion A):
        # el marcador mas frecuente de entre las simulaciones donde gana el que pasa.
        winning_outcome = "home_win" if advancer == home else "away_win"
        adv_scores = [s for s, r in zip(scores, outcomes) if r == winning_outcome]
        if adv_scores:
            hg, ag = Counter(adv_scores).most_common(1)[0][0]
        else:
            hg, ag = (1, 0) if advancer == home else (0, 1)
        result["score"] = f"{hg}-{ag}"

    return result

def simulate_deterministic_group(teams, ratings, params, num_sims=1000):
    table = {t: {"pts": 0, "gf": 0, "ga": 0, "gd": 0} for t in teams}
    match_results = []
    
    for i in range(len(teams)):
        for j in range(i+1, len(teams)):
            home, away = teams[i], teams[j]
            match_data = get_match_consensus(home, away, ratings, params, is_knockout=False, num_sims=num_sims)
            match_results.append(match_data)
            
            # Sumar puntos y goles a la tabla basándonos en el marcador de consenso
            hg, ag = map(int, match_data["score"].split("-"))
            table[home]["gf"] += hg; table[home]["ga"] += ag
            table[away]["gf"] += ag; table[away]["ga"] += hg
            table[home]["gd"] = table[home]["gf"] - table[home]["ga"]
            table[away]["gd"] = table[away]["gf"] - table[away]["ga"]
            
            if hg > ag: table[home]["pts"] += 3
            elif ag > hg: table[away]["pts"] += 3
            else: table[home]["pts"] += 1; table[away]["pts"] += 1
            
    # Ordenar grupo (FIFA format)
    ranked = sorted(table.items(), key=lambda x: (x[1]["pts"], x[1]["gd"], x[1]["gf"], ratings[x[0]]["elo"]), reverse=True)
    return ranked, match_results

def play_deterministic_knockout(matches, ratings, params, num_sims=1000):
    results = {}
    for mid, (ta, tb) in matches.items():
        match_data = get_match_consensus(ta, tb, ratings, params, is_knockout=True, num_sims=num_sims)
        results[mid] = match_data
    return results

In [38]:
def generate_perfect_bracket(groups, ratings, params, mapping_df, num_sims=1000):
    print("Creando el Cuadro Determinista del Mundial 2026...")
    bracket = {"group_stage": {}, "knockouts": {}}
    
    # --- 1. FASE DE GRUPOS ---
    gs_winners, gs_runners, gs_thirds = {}, {}, []
    for group_name, teams in groups.items():
        ranked, matches = simulate_deterministic_group(teams, ratings, params, num_sims)
        
        # Guardamos en el bracket para el Streamlit
        standings = [{"team": t, **stats} for t, stats in ranked]
        bracket["group_stage"][group_name] = {"standings": standings, "matches": matches}
        
        # Preparamos clasificados
        gs_winners[group_name] = ranked[0][0]
        gs_runners[group_name] = ranked[1][0]
        gs_thirds.append({"group": group_name, "team": ranked[2][0], "pts": ranked[2][1]["pts"], 
                          "gd": ranked[2][1]["gd"], "gf": ranked[2][1]["gf"]})
        
    gs_data = {"winners": gs_winners, "runners_up": gs_runners}
    best_thirds = select_best_thirds(gs_thirds, ratings)
    mapping = get_third_place_mapping(best_thirds, mapping_df)
    
    print("Fase de grupos completada. Jugando eliminatorias...")
    
    # --- 2. ELIMINATORIAS ---
    # Diccionario auxiliar para sacar solo los ganadores ("advances") y pasarlos a la siguiente ronda
    def get_adv(results_dict): return {mid: res["advances"] for mid, res in results_dict.items()}

    # 1/16 Final
    r32_matches = build_round_of_32(gs_data, best_thirds, mapping)
    bracket["knockouts"]["round_32"] = play_deterministic_knockout(r32_matches, ratings, params, num_sims)
    
    # Octavos
    r16_matches = build_round_of_16(get_adv(bracket["knockouts"]["round_32"]))
    bracket["knockouts"]["round_16"] = play_deterministic_knockout(r16_matches, ratings, params, num_sims)
    
    # Cuartos
    qf_matches = build_quarters(get_adv(bracket["knockouts"]["round_16"]))
    bracket["knockouts"]["quarter_finals"] = play_deterministic_knockout(qf_matches, ratings, params, num_sims)
    
    # Semis
    sf_matches = build_semis(get_adv(bracket["knockouts"]["quarter_finals"]))
    bracket["knockouts"]["semi_finals"] = play_deterministic_knockout(sf_matches, ratings, params, num_sims)
    
    # Final
    final_match = build_final(get_adv(bracket["knockouts"]["semi_finals"]))
    bracket["knockouts"]["final"] = play_deterministic_knockout(final_match, ratings, params, num_sims)
    
    # Campeón
    bracket["champion"] = bracket["knockouts"]["final"][104]["advances"]
    print(f"¡Simulación terminada! El Campeón del Mundo es: {bracket['champion']}")
    
    return bracket

# Ejecutamos la función mágica
perfect_bracket = generate_perfect_bracket(groups, latest_ratings, optimal_params, third_mapping_df, num_sims=1000)

# Exportamos a JSON
import os
os.makedirs("../data/processed", exist_ok=True)
json_path = "../data/processed/bracket_2026.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(perfect_bracket, f, indent=4, ensure_ascii=False)

print(f"\n✅ ¡Documento generado con éxito! Guardado en: {json_path}")

Creando el Cuadro Determinista del Mundial 2026...
Fase de grupos completada. Jugando eliminatorias...
¡Simulación terminada! El Campeón del Mundo es: Argentina

✅ ¡Documento generado con éxito! Guardado en: ../data/processed/bracket_2026.json


In [54]:
a = df_hist[(df_hist['home_team'] == 'Belgium') | (df_hist['away_team'] == 'Belgium')]
a.tail(30)

,date,home_team,away_team,home_score,away_score,home_elo,away_elo,home_o,home_d,away_o,away_d,home_form,away_form,tournament,neutral,home_o_norm,home_d_norm,away_o_norm,away_d_norm
46519,2023-10-13,Austria,Belgium,2,3,1615.499478,1741.477595,48.935958,26.838257,160.944381,8.607723,0.197026,0.459489,UEFA Euro qualification,False,2.129657,1.475187,7.452055,0.431050
46580,2023-10-16,Belgium,Sweden,1,1,1750.190334,1554.506221,160.618727,8.638105,83.528129,15.831755,0.520894,-0.404586,UEFA Euro qualification,False,6.990010,0.474801,3.867524,0.792809
46649,2023-11-15,Belgium,Serbia,1,0,1741.614168,1633.947218,160.174009,8.666567,66.792670,18.404570,0.505258,0.017160,Friendly,False,6.970656,0.476365,3.092637,0.921648
46745,2023-11-19,Belgium,Azerbaijan,5,0,1743.171880,1430.902646,161.534276,8.620894,15.781786,77.519723,0.505258,0.086395,UEFA Euro qualification,False,7.029854,0.473855,0.730729,3.881963
47088,2024-03-23,Republic of Ireland,Belgium,0,0,1485.647562,1749.226391,42.601752,29.882828,163.221104,8.544542,-0.401976,0.557478,Friendly,False,1.853997,1.642535,7.557472,0.427886
47127,2024-03-26,England,Belgium,2,2,1768.831638,1744.191016,159.302312,7.384401,162.251569,8.592672,0.556099,0.557478,Friendly,False,6.932721,0.405890,7.512580,0.430296
47212,2024-06-05,Belgium,Montenegro,2,0,1742.029494,1477.489976,162.275816,8.592759,38.237010,37.746930,0.557478,-0.174845,Friendly,False,7.062126,0.472308,1.770452,1.890257
47284,2024-06-08,Belgium,Luxembourg,3,0,1742.121326,1451.365544,163.525237,8.552473,13.251130,83.083640,0.557478,0.051995,Friendly,False,7.116499,0.470094,0.613554,4.160588
47384,2024-06-17,Belgium,Slovakia,0,1,1742.262782,1534.647442,163.811897,8.531406,37.851615,34.891190,0.557478,0.555476,UEFA Euro,True,7.128975,0.468936,1.752607,1.747250
47409,2024-06-22,Belgium,Romania,2,0,1700.866297,1607.013378,156.950834,9.524037,58.991160,21.446151,0.276360,0.735652,UEFA Euro,True,6.830386,0.523497,2.731412,1.073961
